# TrashCan Plastic Detector — Colab Training

Self-contained: doesn't need the rest of this repo, only `dataset.zip` uploaded to your
Google Drive. Run cells top to bottom.

**Before running:** upload `data/raw/trashcan/dataset.zip` (527MB, from your local project) to
your Google Drive at `MyDrive/trashcan/dataset.zip`.

In [ ]:
!nvidia-smi
!pip install -q ultralytics scikit-learn pyyaml tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATASET_ZIP = '/content/drive/MyDrive/trashcan/dataset.zip'
assert __import__('os').path.exists(DATASET_ZIP), f'Upload dataset.zip to {DATASET_ZIP} first'

In [ ]:
import zipfile, os
os.makedirs('/content/raw', exist_ok=True)
with zipfile.ZipFile(DATASET_ZIP) as z:
    z.extractall('/content/raw')
RAW = '/content/raw/dataset/material_version'
print(os.listdir(RAW))

## Validate, leak-free split (grouped by source video id), COCO -> YOLO-seg conversion
Same logic as `src/data/prepare_dataset.py` in the main project, inlined here so this
notebook has no dependency on the rest of the repo. See that file / README.md for why we
re-split instead of using TrashCan's own train/val split (it isn't leak-free: the official
val set reuses videos that also appear in the official train set).

In [ ]:
import json, re, shutil
import numpy as np, pandas as pd, yaml
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
SPLIT_FRACTIONS = {'train': 0.70, 'val': 0.15, 'test': 0.15}
CLASS_NAMES = ['rov','plant','animal_fish','animal_starfish','animal_shells','animal_crab',
               'animal_eel','animal_etc','trash_etc','trash_fabric','trash_fishing_gear',
               'trash_metal','trash_paper','trash_plastic','trash_rubber','trash_wood']
VIDEO_RE = re.compile(r'vid_(\d+)_frame(\d+)\.jpg$')
PROCESSED = Path('/content/processed')

def load_coco(p):
    return json.load(open(p))

train_coco = load_coco(f'{RAW}/instances_train_trashcan.json')
val_coco = load_coco(f'{RAW}/instances_val_trashcan.json')
assert [c['name'] for c in train_coco['categories']] == CLASS_NAMES
cat_id_to_yolo = {c['id']: i for i, c in enumerate(train_coco['categories'])}

images_by_id, anns_by_image = {}, {}
for coco, img_dir in [(train_coco, f'{RAW}/train'), (val_coco, f'{RAW}/val')]:
    for img in coco['images']:
        img = dict(img)
        img['path'] = Path(img_dir) / img['file_name']
        m = VIDEO_RE.match(img['file_name'])
        img['video_id'] = m.group(1) if m else None
        images_by_id[img['id']] = img
    for a in coco['annotations']:
        anns_by_image.setdefault(a['image_id'], []).append(a)

valid_ids = [i for i, im in images_by_id.items() if im['path'].exists()]
print(f'{len(valid_ids)} valid images')

In [ ]:
# Leak-free group split by video id (no cross-video duplicate check here -- Colab has no
# local raw images to hash cheaply; the main repo's prepare_dataset.py does that check).
video_ids = [images_by_id[i]['video_id'] or f'novideo_{i}' for i in valid_ids]
ids = np.array(valid_ids); groups = np.array(video_ids)

vt = SPLIT_FRACTIONS['val'] + SPLIT_FRACTIONS['test']
gss1 = GroupShuffleSplit(n_splits=1, test_size=vt, random_state=SEED)
tr_idx, rest_idx = next(gss1.split(ids, groups=groups))
rest_ids, rest_groups = ids[rest_idx], groups[rest_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=SPLIT_FRACTIONS['test']/vt, random_state=SEED)
val_idx, test_idx = next(gss2.split(rest_ids, groups=rest_groups))

assignment = {i: 'train' for i in ids[tr_idx]}
assignment.update({i: 'val' for i in rest_ids[val_idx]})
assignment.update({i: 'test' for i in rest_ids[test_idx]})
print(pd.Series(assignment.values()).value_counts())

In [ ]:
def poly_area(p):
    xs, ys = p[0::2], p[1::2]
    return abs(sum(xs[i]*ys[(i+1)%len(xs)] - xs[(i+1)%len(xs)]*ys[i] for i in range(len(xs)))) / 2

def coco_to_yolo_lines(anns, w, h):
    lines = []
    for a in anns:
        seg = a.get('segmentation')
        parts = [p for p in (seg or []) if isinstance(p, list) and len(p) >= 6]
        if not parts or a.get('area', 0) <= 0:
            continue
        poly = max(parts, key=poly_area)
        cls = cat_id_to_yolo[a['category_id']]
        norm = []
        for i in range(0, len(poly), 2):
            norm += [f"{min(max(poly[i]/w,0),1):.6f}", f"{min(max(poly[i+1]/h,0),1):.6f}"]
        lines.append(f"{cls} " + ' '.join(norm))
    return lines

for split in ['train', 'val', 'test']:
    (PROCESSED / split / 'images').mkdir(parents=True, exist_ok=True)
    (PROCESSED / split / 'labels').mkdir(parents=True, exist_ok=True)

for img_id in valid_ids:
    img = images_by_id[img_id]
    split = assignment[img_id]
    lines = coco_to_yolo_lines(anns_by_image.get(img_id, []), img['width'], img['height'])
    shutil.copyfile(img['path'], PROCESSED / split / 'images' / img['file_name'])
    (PROCESSED / split / 'labels' / (Path(img['file_name']).stem + '.txt')).write_text('\n'.join(lines))

data_yaml = {
    'path': str(PROCESSED), 'train': 'train/images', 'val': 'val/images', 'test': 'test/images',
    'names': {i: n for i, n in enumerate(CLASS_NAMES)},
}
yaml.safe_dump(data_yaml, open(PROCESSED / 'data.yaml', 'w'), sort_keys=False)
print('data.yaml written:', data_yaml)

## Train

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n-seg.pt')
results = model.train(
    data=str(PROCESSED / 'data.yaml'), imgsz=640, batch=16, epochs=80, device=0,
    seed=SEED, patience=15, project='runs/segment', name='trashcan_yolo_seg', plots=True,
)

## Save the trained model back to Drive
So you can download `best_model.pt` and drop it into `models/plastic_detector/` in the main
project to use `predict.py`, `evaluate.py` and `app.py` locally (those only need the .pt file,
not a GPU).

In [ ]:
import shutil
best = Path(results.save_dir) / 'weights' / 'best.pt'
dest_dir = Path('/content/drive/MyDrive/trashcan')
dest_dir.mkdir(parents=True, exist_ok=True)
shutil.copyfile(best, dest_dir / 'best_model.pt')
print('Saved to', dest_dir / 'best_model.pt', '-- download this and place it at models/plastic_detector/best_model.pt locally')